In [1]:
# Load packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("darkgrid")

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Lasso

from sklearn.neighbors import KNeighborsClassifier

from sklearn.decomposition import PCA

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split, cross_val_score


from sklearn.cluster import KMeans
from sklearn.svm import LinearSVC

from sklearn.pipeline import make_pipeline

from sklearn import metrics
from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    mean_absolute_percentage_error,
    confusion_matrix,
    make_scorer,
    precision_score,
    recall_score,
    RocCurveDisplay,
    roc_auc_score,
    r2_score
)

df_all = pd.read_csv('C:/Users/Dell/OneDrive/Documents/Dissertation/diabetic_data.csv')
df_all['age'] = df_all['age'].str.replace('[','(',regex=False)
df_all

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,(0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,(10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,(20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,(30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,(40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,(70-80),?,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,(80-90),?,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,(70-80),?,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,(80-90),?,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [2]:
df_all.shape

(101766, 50)

In [3]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [4]:
df_all.isnull().sum()

encounter_id                0
patient_nbr                 0
race                        0
gender                      0
age                         0
weight                      0
admission_type_id           0
discharge_disposition_id    0
admission_source_id         0
time_in_hospital            0
payer_code                  0
medical_specialty           0
num_lab_procedures          0
num_procedures              0
num_medications             0
number_outpatient           0
number_emergency            0
number_inpatient            0
diag_1                      0
diag_2                      0
diag_3                      0
number_diagnoses            0
max_glu_serum               0
A1Cresult                   0
metformin                   0
repaglinide                 0
nateglinide                 0
chlorpropamide              0
glimepiride                 0
acetohexamide               0
glipizide                   0
glyburide                   0
tolbutamide                 0
pioglitazo

In [13]:
cols = ['race','gender','weight','readmitted']
for i in cols:
    print(i)
    print(df_all[i].unique())

race
['Caucasian' 'AfricanAmerican' '?' 'Other' 'Asian' 'Hispanic']
gender
['Female' 'Male' 'Unknown/Invalid']
weight
['?' '[75-100)' '[50-75)' '[0-25)' '[100-125)' '[25-50)' '[125-150)'
 '[175-200)' '[150-175)' '>200']
readmitted
['NO' '>30' '<30']


In [11]:
df_all['gender'].value_counts()

Female             54708
Male               47055
Unknown/Invalid        3
Name: gender, dtype: int64

In [10]:
df_all['race'].value_counts()

Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: race, dtype: int64

In [47]:
df_new = df_all[['race', 'gender', 'age','weight','time_in_hospital', 'num_lab_procedures','num_procedures','num_medications','number_outpatient','number_emergency','number_inpatient','diag_1','diag_2','diag_3', 'number_diagnoses','max_glu_serum','A1Cresult','metformin','insulin','acarbose','repaglinide','pioglitazone','change','diabetesMed','readmitted' ]]  # Selecting specific columns
df_new

,race,gender,age,weight,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,max_glu_serum,A1Cresult,metformin,insulin,acarbose,repaglinide,pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,(0-10),?,1,41,0,1,0,0,...,None,None,No,No,No,No,No,No,No,NO
1,Caucasian,Female,(10-20),?,3,59,0,18,0,0,...,None,None,No,Up,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,(20-30),?,2,11,5,13,2,0,...,None,None,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,(30-40),?,2,44,1,16,0,0,...,None,None,No,Up,No,No,No,Ch,Yes,NO
4,Caucasian,Male,(40-50),?,1,51,0,8,0,0,...,None,None,No,Steady,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,AfricanAmerican,Male,(70-80),?,3,51,0,16,0,0,...,None,>8,Steady,Down,No,No,No,Ch,Yes,>30
101762,AfricanAmerican,Female,(80-90),?,5,33,3,18,0,0,...,None,None,No,Steady,No,No,No,No,Yes,NO
101763,Caucasian,Male,(70-80),?,1,53,0,9,1,0,...,None,None,Steady,Down,No,No,No,Ch,Yes,NO
101764,Caucasian,Female,(80-90),?,10,45,2,21,0,0,...,None,None,No,Up,No,No,Steady,Ch,Yes,NO


In [48]:
cols = ['race','gender','weight','readmitted','age']
for i in cols:
    print(i)
    print(df_new[i].unique())

race
['Caucasian' 'AfricanAmerican' '?' 'Other' 'Asian' 'Hispanic']
gender
['Female' 'Male' 'Unknown/Invalid']
weight
['?' '[75-100)' '[50-75)' '[0-25)' '[100-125)' '[25-50)' '[125-150)'
 '[175-200)' '[150-175)' '>200']
readmitted
['NO' '>30' '<30']
age
['(0-10)' '(10-20)' '(20-30)' '(30-40)' '(40-50)' '(50-60)' '(60-70)'
 '(70-80)' '(80-90)' '(90-100)']


In [49]:
df_new['race'].value_counts()

Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: race, dtype: int64

In [50]:
df_new['gender'].value_counts()

Female             54708
Male               47055
Unknown/Invalid        3
Name: gender, dtype: int64

In [51]:
df_new['age'].value_counts()

(70-80)     26068
(60-70)     22483
(50-60)     17256
(80-90)     17197
(40-50)      9685
(30-40)      3775
(90-100)     2793
(20-30)      1657
(10-20)       691
(0-10)        161
Name: age, dtype: int64

In [52]:
df_new['weight'].value_counts()

?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: weight, dtype: int64

In [54]:
#### need to replace '?' with 'Other' in "race" column ####

In [33]:
# Modify the DataFrame using .loc
row_indexer = df_new['race'] = '?'   # Example condition for row indexing
col_indexer = 'race'  # Example column name for column indexing
value = 'Other'  # Example value to set

df_new.loc[row_indexer, col_indexer] = value


C:\Users\Dell\AppData\Local\Temp\ipykernel_23592\1551156634.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  row_indexer = df_new['race'] = '?'   # Example condition for row indexing
C:\Users\Dell\AppData\Local\Temp\ipykernel_23592\1551156634.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new.loc[row_indexer, col_indexer] = value
